# Геймификация на учебной платформе
Сравниваем старый интерфейс A с системой достижений B за 4–24 мая 2026 года. Единица анализа — ученик. Гипотеза для каждой метрики: вероятность события одинакова в обеих группах. Проверки двусторонние.

In [1]:
from pathlib import Path
from math import sqrt, erfc
from statistics import NormalDist
import pandas as pd

users = pd.read_csv(Path('data/ab_test_users_groups.csv'), parse_dates=['signup_date'])
events = pd.read_csv(Path('data/ab_test_events.csv'), parse_dates=['timestamp'])
users.info()
print(events.event_type.value_counts())
print('Пропуски:', users.isna().sum().to_dict(), events.isna().sum().to_dict())
print('Полные дубли:', users.duplicated().sum(), events.duplicated().sum())
conflicts = users.groupby('user_id').group.nunique()
conflicts = conflicts[conflicts > 1].index
print('Пересечения:', len(conflicts))
print('События без участника:', (~events.user_id.isin(users.user_id)).sum())
users = users.drop_duplicates().loc[lambda x: ~x.user_id.isin(conflicts)].copy()
assert users.user_id.is_unique and set(users.group) == {'A','B'}
counts = users.group.value_counts()
# Критерий Пирсона с одной степенью свободы для предполагаемого сплита 1:1
chi2 = ((counts - counts.sum()/2)**2 / (counts.sum()/2)).sum()
print('Распределение:', counts.to_dict(), 'SRM p =', round(erfc(sqrt(chi2/2)), 4))
events = events.drop_duplicates().merge(users[['user_id','group']], on='user_id', validate='many_to_one')
activity = events.user_id.value_counts()
print('Наиболее активные участники:', activity.head(15).tolist())
outliers = activity[activity > 100].index
print('Исключены из основного анализа:', len(outliers))

<class 'pandas.DataFrame'>
RangeIndex: 6008 entries, 0 to 6007
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   user_id      6008 non-null   str           
 1   group        6008 non-null   str           
 2   signup_date  6008 non-null   datetime64[us]
 3   grade        6008 non-null   int64         
dtypes: datetime64[us](1), int64(1), str(2)
memory usage: 187.9 KB
event_type
login                 75552
homework_started      68317
homework_completed    28995
return_visit_d7        4519
return_visit_d14       1947
Name: count, dtype: int64
Пропуски: {'user_id': 0, 'group': 0, 'signup_date': 0, 'grade': 0} {'user_id': 0, 'timestamp': 0, 'event_type': 0}
Полные дубли: 0 2
Пересечения: 8
События без участника: 0
Распределение: {'A': 2997, 'B': 2995} SRM p = 0.9794
Наиболее активные участники: [842, 829, 793, 790, 780, 771, 770, 765, 760, 738, 729, 711, 52, 51, 51]
Исключены из основного анализа: 12


## Метрики и фильтрация
Порог 100 уникальных событий выбран по разрыву в распределении активности. Это признак аномалии, а не доказательство автоматизации. Ниже повторяем оценку со всеми участниками. Удержание измеряем по готовым флагам возврата. Завершение хотя бы одного ДЗ служит отдельной метрикой: события завершения курса в выгрузке нет.

In [2]:
flags = pd.crosstab(events.user_id, events.event_type).gt(0)
full = users.set_index('user_id').join(flags)
metrics = ['homework_completed', 'return_visit_d7', 'return_visit_d14']
for column in flags:
    full[column] = full[column].eq(True)
sample = full.drop(index=outliers)
def compare(data):
    rows = []
    for metric in metrics:
        a, b = [data.loc[data.group == g, metric] for g in ['A', 'B']]
        pa, pb = a.mean(), b.mean()
        delta = pb - pa
        pooled = (a.sum()+b.sum())/(len(a)+len(b))
        se_null = sqrt(pooled*(1-pooled)*(1/len(a)+1/len(b)))
        se = sqrt(pa*(1-pa)/len(a)+pb*(1-pb)/len(b))
        p = erfc(abs(delta/se_null)/sqrt(2))
        mde = (NormalDist().inv_cdf(.975)+NormalDist().inv_cdf(.8))*sqrt(pa*(1-pa)*(1/len(a)+1/len(b)))
        rows.append([pa*100,pb*100,delta*100,(delta-1.96*se)*100,(delta+1.96*se)*100,p,min(1,3*p),100*mde])
    return pd.DataFrame(rows,index=metrics,columns=['A %','B %','Δ п.п.','ДИ низ','ДИ верх','p','p Bonf','MDE п.п.'])
result = compare(sample)
print(result.to_string(float_format=lambda x: f'{x:.5g}'))
print('Все участники, разница в п.п.:')
print(compare(full)['Δ п.п.'])

                      A %    B %  Δ п.п.  ДИ низ  ДИ верх          p     p Bonf  MDE п.п.
homework_completed 97.627 98.829  1.2016  0.5336   1.8697 0.00042978  0.0012893    1.1029
return_visit_d7    72.694 77.979  5.2847  3.1038   7.4656 2.1346e-06 6.4037e-06    3.2282
return_visit_d14    28.81 36.245  7.4348  5.0675   9.8022 8.4436e-10 2.5331e-09    3.2814
Все участники, разница в п.п.:
homework_completed    1.200421
return_visit_d7       5.257233
return_visit_d14      7.464950
Name: Δ п.п., dtype: float64


Для бинарных метрик применяем z-тест двух долей, обычный 95% интервал разницы и поправку Бонферрони для трёх тестов. MDE — приближение при мощности 80% и уровне 5% для одного теста.

In [3]:
# Последовательная воронка: каждое следующее событие должно идти позже предыдущего.
steps = ['login','homework_started','homework_completed']
eligible = events[events.user_id.isin(sample.index)]
previous = None
funnel = {}
for step in steps:
    stage = eligible.loc[eligible.event_type == step, ['user_id','timestamp']]
    if previous is not None:
        stage = stage.join(previous.rename('previous'), on='user_id')
        stage = stage[stage.timestamp > stage.previous]
    previous = stage.groupby('user_id').timestamp.min()
    funnel[step] = sample.index.isin(previous.index)
rates = pd.DataFrame(funnel,index=sample.index).groupby(sample.group).mean()*100
print(rates.round(2))
print('Завершение ДЗ по классам, %:')
print((sample.groupby(['grade','group']).homework_completed.mean().unstack()*100).round(1))

       login  homework_started  homework_completed
group                                             
A      100.0             100.0               97.63
B      100.0             100.0               98.83
Завершение ДЗ по классам, %:
group     A      B
grade             
5      97.5  100.0
6      97.3  100.0
7      97.8  100.0
8      97.6   99.5
9      97.9   99.5
10     97.6   97.0
11     97.8   95.8


![Последовательная воронка](funnel.png)

## Решение
Завершили хотя бы одно ДЗ
A 97.6%     B 98.8%     Разница +1.2 п.п.
95% ДИ [+0.5; +1.9] п.п.   •   p с поправкой 0.00129

Вернулись на 7-й день
A 72.7%     B 78.0%     Разница +5.3 п.п.
95% ДИ [+3.1; +7.5] п.п.   •   p с поправкой 6.4e-06

Вернулись на 14-й день
A 28.8%     B 36.2%     Разница +7.4 п.п.
95% ДИ [+5.1; +9.8] п.п.   •   p с поправкой 2.53e-09

Продолжить проверку перед общим запуском. Добавить событие завершения курса и заранее согласовать минимальный эффект, который оправдает внедрение.